# Lecture 4: Python Review 2

## Key Concepts Review

### Binary Numbers and Float Precision

Computers store numbers in binary (base 2), which can cause precision issues with decimal numbers:

```python
# Float precision issues
print(0.1 + 0.2)  # Output: 0.30000000000000004 (not exactly 0.3!)
print(0.1 + 0.2 == 0.3)  # Output: False

# Better way to compare floats
def close_enough(a, b, tolerance=1e-9):
    return abs(a - b) < tolerance

print(close_enough(0.1 + 0.2, 0.3))  # Output: True
```

### Objects and Types
Everything in Python (except keywords) is an object with a type:

```python
# Scalar types (single values)
concentration = 0.5      # float
num_atoms = 6           # int
element = "H"           # string
is_stable = True        # boolean

# Non-scalar types (collections)
concentrations = [0.1, 0.2, 0.3]     # list
compound_data = {"name": "water", "formula": "H2O"}  # dictionary

# Check types
print(type(concentration))  # <class 'float'>
print(type(concentrations)) # <class 'list'>

# Type comparison
print(isinstance(concentration, float)) # True
print(isinstance(concentration, list)) # False
```


### Useful Operators

```python
# Comparison operators that handle precision
import math
math.isclose(0.1 + 0.2, 0.3)  # Better than == for floats

# Membership operators
"H" in ["H", "O", "N"]  # True
"formula" in {"name": "water", "formula": "H2O"}  # True
```

## Exercise 1: Concentration Precision Checker

**Objective:** Understand float precision issues when working with chemical concentrations.

**Problem:** In analytical chemistry, you often need to compare measured concentrations, but computer precision can cause issues. Write a function that safely compares concentrations and handles rounding errors.

**Your Task:** Create a function called `concentrations_equal()` that:

- Takes two concentration values and an optional tolerance (default 1e-6)
- Returns True if the concentrations are "close enough" considering float precision

Also create a function `round_to_significant_figures()` that rounds a number to a given number of significant figures

**Hints:**
- Use abs(a - b) < tolerance for comparison
- For significant figures, you might need to use logarithms: import math then math.log10()
- Think about how to handle very small numbers (like 0.00123)


In [ ]:
def concentrations_equal(conc1: float, conc2: float, tolerance: float = 1E-6) -> bool:
    """
    Return True if two concentrations are close enough to be considered equal,
    allowing for small floating-point precision errors. 
    """
    return abs(conc1 - conc2) <= tolerance  

In [ ]:
import math

def round_to_significant_figures(value: float, precision: int) -> float:    # Define function with parameters value and precision
    if precision <= 0:                                                      # Check for non-positive precision
        raise ValueError("precision must be a positive integer")            # Raise an error if precision is not a positive integer 
    if value == 0: # Check for zero value
        return 0.0

    sign = -1 if value < 0 else 1                                           # Determines the sign of the number to preserve it after rounding. 
    abs_value = abs(value)                                                  # Removes the sign so the calculation can be done on the magnitude of the number. Logarithm of the absolute value is rounded down.
    exponent = math.floor(math.log10(abs_value))                            # Calculates the exponent of the number in scientific notation.
    
    
    factor = 10 ** (precision - 1 - exponent)                               # Calculates the factor to scale the number so that the desired number of significant figures can be rounded correctly.

    rounded = round(abs_value * factor) / factor                            # Rounds the scaled number to the nearest integer and then scales it back down to its original magnitude, preserving the desired number of significant figures.
    return sign * rounded

### Test Cases: Concentration comparison


In [6]:
print(concentrations_equal(0.1 + 0.2, 0.3))           # Should be True
print(concentrations_equal(0.1001, 0.1002, 1e-3))     # Should be True  
print(concentrations_equal(0.1001, 0.1002, 1e-5))     # Should be False

True
True
False


### Test Cases: Rounding

In [7]:
print(round_to_significant_figures(0.123456, 3))       # Should be 0.123
print(round_to_significant_figures(1234.56, 3))        # Should be 1230.0
print(round_to_significant_figures(0.00123456, 3))     # Should be 0.00123

0.123
1230.0
0.00123


## Exercise 2: Reaction Optimization Calculator

**Objective:** Use optimization thinking to find the best reaction conditions while working with different data types.

**Problem:** You're optimizing a chemical reaction by testing different temperatures and catalyst amounts. Write a program that finds the optimal conditions from experimental data.

**Your Task:** Create functions that:

- Store experimental data as a list of dictionaries
- Find the conditions that give maximum yield
- Find all conditions that give yield within a certain percentage of the maximum
- Calculate the cost-effectiveness (yield per unit cost) for each condition

**Data Structure:**

In [8]:
# Each experiment is a dictionary
experimental_data = [
    {"temperature": 25.0, "catalyst": 0.1, "yield": 45.2, "cost": 10.5},
    {"temperature": 50.0, "catalyst": 0.1, "yield": 67.8, "cost": 12.0},
    {"temperature": 75.0, "catalyst": 0.1, "yield": 82.3, "cost": 15.5},
    {"temperature": 100.0, "catalyst": 0.1, "yield": 78.9, "cost": 20.0},
    {"temperature": 50.0, "catalyst": 0.2, "yield": 75.4, "cost": 18.0},
    {"temperature": 75.0, "catalyst": 0.2, "yield": 89.1, "cost": 22.5}
]

**Functions to Write:** 

In [ ]:
from typing import List, Dict
# Backwards-compatible -- for Python 3.8 and higher can just use list, dict below, no import
def find_max_yield_condition(data: List[Dict[str, float]]) -> Dict[str, float]:             # Define function with parameter data, which is a list of dictionaries with dictionary keys as strings and values as floats
    max_yield_experiment = max(data, key=lambda x: x["yield"])                              # max function is used to find the dictionary with the maximum yield value in the list of dictionaries. 
    
    """
    The lambda function takes an argument x, which represents each dictionary in the list, and returns the value associated with the "yield" key in that dictionary. 
    Lambda functions are anonymous functions defined using the lambda keyword, and they are often used for short, simple operations like this one. 
    The max function then compares these yield values across all dictionaries and returns the dictionary with the highest yield.
    """

    return max_yield_experiment                                                             # Return the dictionary of the experiment with the maximum yield
    

print(find_max_yield_condition(experimental_data))                           

{'temperature': 75.0, 'catalyst': 0.2, 'yield': 89.1, 'cost': 22.5}


In [10]:
from typing import List, Dict

def find_conditions_near_optimum(data: List[Dict[str, float]], percentage: int = 10) -> List[Dict[str, float]]:
    # Return all conditions within 10% of max yield
    max_yield = max(data, key=lambda x: x["yield"])["yield"]                                       # Find the maximum yield value in the list of dictionaries
    lower_bound = max_yield * (1 - percentage / 100)                                               # Define variable lower_bound and calculate the lower bound of the yield range based on the specified percentage
    upper_bound = max_yield * (1 + percentage / 100)                                               # Define variable upper_bound and calculate the upper bound of the yield range based on the specified percentage
    return [condition for condition in data if lower_bound <= condition["yield"] <= upper_bound]   # Return a list of dictionaries containing the conditions that fall within the specified yield range

print(find_conditions_near_optimum(experimental_data, percentage=10))                               #List comprehension that takes each condition in data and checks if its yield is within the lower and upper bounds. 
                                                                                                    #If it is, the condition is included in the returned list.

#The variable condition is created automatically by the comprehension itself and thus does not have to be defined explicitly. It is a temporary name for "the current dictionary being checked".
#It takes the first dictionary in data and calls it condition. 
#Then it checks if the yield of that condition is within the lower and upper bounds. If it is, the condition is included in the returned list.
#Then it moves to the next dictionary and repeats the process until all dictionaries in data have been checked.     


[{'temperature': 75.0, 'catalyst': 0.1, 'yield': 82.3, 'cost': 15.5}, {'temperature': 75.0, 'catalyst': 0.2, 'yield': 89.1, 'cost': 22.5}]


In [13]:
from typing import List, Dict

def calculate_cost_effectiveness(data: List[Dict[str, float]]) -> List[Dict[str, float]]:
    # Add "efficiency" key to each dictionary (yield/cost)
    # Return the modified data
    efficiency_list = []                                                # Create an empty list to store the modified dictionaries
    for condition in data:                                              # Iterate over each dictionary in the input list
        efficiency = round(condition["yield"] / condition["cost"], 2)   # Calculate the efficiency as yield divided by cost and round it to 2 decimal places
        modified_condition = condition.copy()                           # Create a copy of the current dictionary to avoid modifying the original
        modified_condition["efficiency"] = efficiency                   # Add the calculated efficiency to the copied dictionary
        efficiency_list.append(modified_condition)                      # Append the modified dictionary to the new list
    return efficiency_list                                              # Return the list of modified dictionaries

In [ ]:
from typing import List, Dict

def optimize_by_criteria(data: List[Dict[str, float]], criteria: str = "yield") -> List[Dict[str, float]]:
    # Return best condition based on "yield", "cost", or "efficiency"
    
    if criteria not in ["yield", "cost", "efficiency"]:  # Check if the criteria is valid
        raise ValueError("Criteria must be 'yield', 'cost', or 'efficiency'")  # Raise an error if the criteria is not valid
    
    return [condition for condition in data if condition.get(criteria) == min(d.get(criteria) for d in data)] 
# Return a list of dictionaries containing the conditions that have the maximum value for the specified criteria. 
# The min function is used to find the minimum value for the specified criteria across all dictionaries in the input list, and then a list comprehension is used to filter the input list to include only those dictionaries that have that minimum value for the specified criteria.

### Test case:

In [15]:
# Basic Optimization
best_yield = find_max_yield_condition(experimental_data)
print(f"Best yield: {best_yield}")
# Expected: Should return the condition with 89.1% yield

near_optimum = find_conditions_near_optimum(experimental_data, 15)
print(f"Conditions within 15% of optimum: {len(near_optimum)} found")
# Expected: Should find multiple conditions within 15% of 89.1%

data_with_efficiency = calculate_cost_effectiveness(experimental_data)
best_efficiency = optimize_by_criteria(data_with_efficiency, "efficiency")
print(f"Most cost-effective: {best_efficiency}")


Best yield: {'temperature': 75.0, 'catalyst': 0.2, 'yield': 89.1, 'cost': 22.5}
Conditions within 15% of optimum: 3 found
Most cost-effective: [{'temperature': 50.0, 'catalyst': 0.1, 'yield': 67.8, 'cost': 12.0, 'efficiency': 5.65}]


## Exercise 3: Reaction Simulator Class

**Objective:** Create a reaction simulator that models the concentration of the reagents and products as a reaction progresses. 

**Problem:** Build a ChemicalReaction class that simulates simple chemical reactions, tracks reactant consumption, and handles errors when invalid data is provided.

**Your Task:** Complete the class below:

In [3]:
import math
from typing import Dict
from collections import defaultdict
from itertools import chain

class ChemicalReaction:
    def __init__(self, reactants: Dict[str, int], products: Dict[str, int], reaction_rate: float = 1.0):
        """
        Initialize a chemical reaction.
        reactants: dict like {"H2": 2, "O2": 1}
        products: dict like {"H2O": 2}
        reaction_rate: how many reaction units happen per second
        """
        self.reactants = reactants                                                      # self.reactants is a dictionary that keeps track of the stoichiometric coefficients of the reactants in the chemical reaction.
        self.products = products                                                        # Same for the products, storing their stoichiometric coefficients.
        self.reaction_rate = reaction_rate                                              # reaction_rate is a float that represents how many reaction units happen per second.

        self.reaction_vessel_content: Dict[str, float] = {                              # self.reaction_vessel_content is a dictionary that creates the internal state of the reaction vessel. Every compound in the reaction is initialized to 0.0 because at the start, nothing has been added yet.
            compound: 0.0 for compound in chain(reactants.keys(), products.keys())      # dictionary comprehension that iterates over the keys of both reactants and products dictionaries, initializing each compound's amount to 0.0 in the reaction vessel by creating a dictionary entry for each compound with a value of 0.0.
        }

    def add_reactant_amount(self, compound: str, amount: float):                        
        """Add the given amount of a compound to the reaction vessel."""
        if compound not in self.reactants and compound not in self.products:            # Check if the compound is part of the reaction (either a reactant or a product). If not, raise a ValueError.
            raise ValueError(f"{compound} is not part of this reaction.")               
        if amount < 0:                                                                  # Check if the amount is negative. If it is, raise a ValueError
            raise ValueError("Amount must be non-negative.")

        self.reaction_vessel_content[compound] += amount                                # Add the specified amount of the compound to the reaction vessel's content. This updates the internal state of the reaction vessel to reflect the new amount of the compound present.

    def run_reaction(self, time_seconds: float = 1.0):
        """
        Run the reaction for the given time.
        The total reaction progress is reaction_rate * time_seconds.
        If there are not enough reactants for that amount, stop with an error.
        """
        if time_seconds < 0:                                                            # Check if the time_seconds is negative. If it is, raise a ValueError
            raise ValueError("time_seconds must be non-negative.")
        if time_seconds == 0:                                                           # If the time_seconds is zero, return immediately without making any changes to the reaction vessel's content. This is a special case where no reaction occurs because no time has passed.
            return

        reaction_units = self.reaction_rate * time_seconds                              # Calculates how much reaction progress should occur in the given time by multiplying the reaction rate by the time in seconds. This gives the total number of reaction units that should be processed.
        max_possible_units = min(                                                       # Calculates the maximum number of reaction units that can occur based on the available amounts of reactants in the reaction vessel. It does this by iterating over each reactant and dividing its current amount in the reaction vessel by its stoichiometric coefficient. The minimum of these values is taken to ensure that the reaction does not proceed beyond what is possible given the available reactants.
            self.reaction_vessel_content[reactant] / coefficient                        
            for reactant, coefficient in self.reactants.items()
            if coefficient > 0
        )

        if reaction_units > max_possible_units:                                         # If the calculated reaction_units exceeds the max_possible_units, it means there are not enough reactants available to complete the requested reaction. In this case, a ValueError is raised with a complementary message 
                raise ValueError(
                "Not enough reactants available to complete the requested reaction."
            )

        for reactant, coefficient in self.reactants.items():                            # Loops over all reactants and theor stoichiometric coefficients
            self.reaction_vessel_content[reactant] -= coefficient * reaction_units      # reduces the amount of each reactant in the reaction vessel by the amount consumed in the reaction, which is calculated as the stoichiometric coefficient multiplied by the number of reaction units.

        for product, coefficient in self.products.items():                              # Loops over all products and their stoichiometric coefficients
            self.reaction_vessel_content[product] += coefficient * reaction_units       # Adds the amount of product created by the reaction 

    def get_reaction_status(self):
        """Return current content of the reaction vessel as a dictionary."""
        return self.reaction_vessel_content                                             # Sends back the dictionary showing all current amounts

### Test cases

In [17]:
water_reaction = ChemicalReaction(
    reactants={"H2": 2, "O2": 1},
    products={"H2O": 2},
    reaction_rate=0.5
)

In [18]:
## Test case 1: General function of the simulator

# Add reactants
water_reaction.add_reactant_amount("H2", 10.0)
water_reaction.add_reactant_amount("O2", 3.0)

# Check initial state of reaction vessel
reaction_status = water_reaction.get_reaction_status() 
print(reaction_status)
assert reaction_status['H2'] == 10.0
assert reaction_status['O2'] == 3.0
assert reaction_status['H2O'] == 0

# Run reaction
water_reaction.run_reaction(2.0)  # Run for 2 timepoints

# Check content of reaction vessel again
reaction_status = water_reaction.get_reaction_status() 
print(reaction_status)
assert reaction_status['H2'] == 8.0
assert reaction_status['O2'] == 2.0
assert reaction_status['H2O'] == 2.0

{'H2': 10.0, 'O2': 3.0, 'H2O': 0.0}
{'H2': 8.0, 'O2': 2.0, 'H2O': 2.0}


In [19]:
## Test case 2: Test error handling when adding compound not in reaction
water_reaction.add_reactant_amount("N2", 10.0)

ValueError: N2 is not part of this reaction.

In [20]:
## Test case 3: Let reaction run to completion
water_reaction.add_reactant_amount("H2", 2.0)
water_reaction.run_reaction(4.0)

# Check content of reaction vessel again
reaction_status = water_reaction.get_reaction_status() 
print(reaction_status)
assert reaction_status['H2'] == 6.0
assert reaction_status['O2'] == 0.0
assert reaction_status['H2O'] == 6.0

{'H2': 6.0, 'O2': 0.0, 'H2O': 6.0}
